# Day 049 · statsmodels 回归
**statsmodels** · 阶段 P2 · Python 量化工具栈

> 前面我们看清了收益是厚尾的。这一节请出回归分析的主角 statsmodels,用它的 sm.OLS().fit().summary() 干一件特别有用的事:把一只票的涨跌拆开看。你以为隆基绿能涨是因为公司好,其实一大半是被大盘抬上去的。我们用回归找一条线:大盘涨 1%,这票大概跟着涨多少,那个放大倍数就叫贝塔;扣掉跟大盘有关的部分,它自己还能多赚或少赚的,叫阿尔法。先跑一个只看市场的单因子模型,再升级到经典的 Fama-French 三因子,把涨跌细分成市场、规模、价值3 股力量。读懂 summary 里的 R方、t 值、p 值,你就再也不会把『大盘普涨的浮盈』错当成『自己选股的真本事』。

---

**课件生成日期:** 2026-06-14  ·  **建议学习时长:** 20 分钟

学习路径建议:1)先看视频建立直觉 → 2)阅读本 notebook 跑代码 → 3)看 PDF 课件复习要点 → 4)做自测题

## 🔧 第一步:环境自检 + 自动安装

**第一次拿到这份 notebook,请先运行下面这一格。** 它会:
1. 检查所有需要的 Python 包,缺什么自动 `pip install` 装上
2. 注入中文字体到 matplotlib(让图标不出乱码)
3. 跑完看到 `✓ 环境就绪` 就可以继续下面的代码

> 这一格只在第一次跑要等几十秒,后面再开 notebook 就秒过。

In [1]:
# === 环境自检 + 自动安装(运行此单元格即可) ===
# 检测缺失的库 → 自动 pip 安装 → 注入中文字体 → 一行命令搞定
import importlib
import subprocess
import sys
import os

REQUIRED = ["baostock", "matplotlib", "numpy", "numpy_financial", "pandas", "scipy", "sklearn", "statsmodels", "yfinance"]
PIP_NAME = {
    "sklearn": "scikit-learn",
    "cv2": "opencv-python",
    "PIL": "Pillow",
    "bs4": "beautifulsoup4",
    "yaml": "PyYAML",
}

missing = []
for mod in REQUIRED:
    try:
        importlib.import_module(mod)
    except ImportError:
        missing.append(PIP_NAME.get(mod, mod))

if missing:
    print(f"⏳ 缺少 {len(missing)} 个包,正在自动安装:{missing}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
    print("✓ 安装完成")
else:
    print(f"✓ 所有 {len(REQUIRED)} 个必需库已就绪")

# === 中文字体配置(让 matplotlib 不出乱码) ===
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

CJK_FONT_PATHS = [
    "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc",  # Linux/WSL
    "C:/Windows/Fonts/msyh.ttc",                               # Windows 微软雅黑
    "C:/Windows/Fonts/simhei.ttf",                             # Windows 黑体
    "/System/Library/Fonts/PingFang.ttc",                      # macOS 苹方
    "/System/Library/Fonts/STHeiti Medium.ttc",                # macOS 黑体
]
for p in CJK_FONT_PATHS:
    if os.path.exists(p):
        fm.fontManager.addfont(p)
        print(f"✓ 中文字体已加载:{os.path.basename(p)}")
        break
plt.rcParams["font.sans-serif"] = ["Noto Sans CJK JP", "Microsoft YaHei",
                                    "PingFang SC", "SimHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
print("✓ 环境就绪 — 现在可以跑下面的代码单元格")


✓ 所有 9 个必需库已就绪
✓ 中文字体已加载:NotoSansCJK-Regular.ttc
✓ 环境就绪 — 现在可以跑下面的代码单元格


## 学习目标

- 理解回归就是找一条线来解释涨跌:大盘涨多少,这票大概跟着涨多少
- 搞懂贝塔和阿尔法:贝塔是跟大盘的放大倍数,阿尔法是跟大盘无关、自己的真本事
- 会读 statsmodels 的 summary:R方看解释了几成,t 值和 p 值看这个系数靠不靠谱
- 会用 sm.OLS().fit() 跑单因子 CAPM,再升级跑 Fama-French 三因子模型
- 知道三因子各代表什么力量:市场 MKT、规模 SMB(小盘减大盘)、价值 HML(价值减成长)

## 历史背景:老雷把大盘普涨的浮盈当成自己选股牛,行情一回头全还回去

老雷 2020 年那波行情里赚得眉飞色舞,账户翻了不少,逢人就说自己眼光毒、选股准。可他没注意到一件事:那一年是大盘整体往上抬,几乎随便买点票都在涨。他自以为的『选股真本事』,其实一大半是被大盘水涨船高带起来的浮盈。后来行情一回头,大盘往下走,他手里那些『牛股』跌得比大盘还狠,之前赚的几乎全数还了回去。老雷想不通:同样是我选的票,怎么涨的时候是我厉害、跌的时候就成了大盘的锅?后来他学了回归才明白,一只票的涨跌得拆开看：有一部分是跟着大盘走的(那是贝塔,大盘带的),还有一部分才是它自己独立的本事(那是阿尔法)。把这两块分清楚,你才知道自己到底是真有水平,还是只是站在了上涨的电梯里。本节就用 statsmodels 把『大盘带的』和『真本事』一刀切开。



## 核心概念

下面每一节是听完视频后回头细读的内容。

### 1. 回归是什么:找一条线来解释涨跌

回归这个词听着高深,其实就是找一条最能解释数据的直线。把隆基绿能每天的涨跌和大盘每天的涨跌画成一堆散点,回归帮你在这堆点里穿一条最贴合的直线。这条线的斜率告诉你:大盘涨 1%,这票大概跟着涨多少。比如斜率是 1.8,意思就是大盘涨 1%、这票平均涨 1.8%。statsmodels 里一句 sm.OLS(y, x).fit() 就能把这条线算出来,y 是隆基的涨跌,x 是大盘的涨跌。


### 2. 贝塔 vs 阿尔法:大盘带的 vs 自己的真本事

那条回归线有两个关键数字。斜率叫贝塔,是这票跟着大盘的放大倍数:贝塔 1.8 就是大盘动 1、它动 1.8,比大盘更猛。截距叫阿尔法,是跟大盘完全无关、这票自己多赚或少赚的部分:大盘原地不动时它还能涨的那一点点就是正阿尔法。说白了贝塔是大盘带的、阿尔法才是自己的真本事。老雷的问题就是把一身贝塔(大盘带的浮盈)误当成了阿尔法(自己的水平)。在 summary 里阿尔法就是那行 const,贝塔就是大盘那一行的系数。


### 3. R方:这条线能解释涨跌的几成

光有一条线还不够,你得知道这条线靠不靠谱。R方就回答这个:它是 0 到 1 之间的一个数,告诉你这条线能解释涨跌的几成。R方 0.55 就是说,隆基的涨跌有 55% 能被大盘解释,剩下 45% 是大盘之外的原因。R方高,说明这票很大程度被大盘牵着走;R方低,说明它有不少自己的脾气。注意 R方高不代表能赚钱,它只说明被大盘解释得多而已,这一点后面坑里会专门讲。summary 里它叫 R-squared。


### 4. 读 summary:t 值和 p 值看系数靠不靠谱

sm.OLS().fit().summary() 会打印一大张表,别被吓到,先盯住两个数:t 值和 p 值。它们回答同一个问题:这个系数是真有东西,还是纯属碰巧?判断很简单,p 值小于 0.05(或者 t 值绝对值大于 2)才当真,否则这个系数可能就是噪声。比如三因子里价值因子的系数算出来不小,但如果它的 p 值是 0.4,那就不能信,等于没解释力。每次看回归结果,先看 p 值过不过关,再看系数大小,顺序别反。


### 5. 三因子:市场之外再加规模和价值两股力量

只用大盘一个因子解释涨跌,往往不够。Fama 和 French 两位学者发现,在市场之外,还有两股力量长期影响股票涨跌:一是规模,小盘股和大盘股表现不同;二是价值,便宜的价值股和贵的成长股表现不同。于是有了三因子。我们这样造:市场因子 MKT 就是沪深300 的涨跌;规模因子 SMB 用中证1000 减沪深300(小盘减大盘);价值因子 HML 用沪深300价值减沪深300成长(价值减成长)。把这3 股力量一起放进回归,就能看出隆基的涨跌里,各有多少是被这3 股力量分别带动的。


## 实操:statsmodels 回归 — sm.OLS().fit().summary() / 单因子 CAPM 看贝塔阿尔法 / Fama-French 三因子拆解隆基绿能涨跌

下面这段代码跟视频里讲解的 highlights 是一致的,可以**直接 Run All** 看结果。

**依赖安装:**
```bash
pip install pandas numpy matplotlib yfinance akshare statsmodels
```


In [2]:
# day_049_statsmodels_regression.py — statsmodels 回归分析:sm.OLS().fit().summary() / 单因子 CAPM / Fama-French 三因子
# 真名上屏:statsmodels.api as sm / sm.add_constant / sm.OLS / .fit() / .summary() / R-squared / t / P>|t|
# 一个反直觉真相:一只票涨,你以为是公司好,其实一大半是被大盘抬上去的(贝塔),只有小部分是它自己的真本事(阿尔法)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import baostock as bs

# ==== 铁律62:数据缓存 helper —— 自适应定位仓库根 data/ 目录 ====
def _data_path(_name):
    # 铁律62:data/ 放在 notebook 文件夹里。仓库根(run_lab)存取 out/notebook/data/;
    # 原版 notebook 在 out/notebook/ 用 data/;中国版在 out/notebook/cn/ 用 ../data/
    from pathlib import Path as _P
    _here = _P.cwd()
    for _b in [_here/'data', _here/'..'/'data', _here/'out'/'notebook'/'data', _here/'..'/'..'/'data', _here/'..'/'..'/'..'/'data']:
        if (_b/_name).exists():
            return str(_b/_name)
    if (_here/'out'/'notebook').exists():
        _t = _here/'out'/'notebook'/'data'
    elif _here.name == 'cn':
        _t = _here/'..'/'data'
    else:
        _t = _here/'data'
    _t.mkdir(parents=True, exist_ok=True)
    return str(_t/_name)

pd.set_option('display.width', 160)
plt.rcParams['axes.unicode_minus'] = False
START, END = '2019-01-01', '2024-12-31'
TRADE_DAYS = 244  # A 股一年约 244 个交易日,用来把日阿尔法年化

# 标的:隆基绿能(光伏龙头,暴涨暴跌故事强,贝塔高)+ 4 个指数(市场/小盘/价值/成长)
CODES = {
    '隆基绿能': 'sh.601012',
    '沪深300':  'sh.000300',   # 市场基准
    '中证1000': 'sh.000852',   # 小盘,做规模因子 SMB
    '沪深300价值': 'sh.000919', # 价值,做价值因子 HML
    '沪深300成长': 'sh.000918', # 成长,做价值因子 HML
}

def fetch(code):
    """拉某个代码的日线收盘价,返回 date,close 两列 DataFrame"""
    rs = bs.query_history_k_data_plus(code, 'date,close', start_date=START, end_date=END, frequency='d')
    rows = []
    while rs.error_code == '0' and rs.next():
        rows.append(rs.get_row_data())
    df = pd.DataFrame(rows, columns=['date', 'close'])
    df['date'] = pd.to_datetime(df['date'])
    df['close'] = pd.to_numeric(df['close'], errors='coerce')
    return df.set_index('date')['close'].sort_index()

# ==== 0. 数据接入:CSV 缓存优先,没有才登录 baostock 联网拉取(铁律62)====
print('==== 0. 拉隆基绿能 + 4 个指数的日线收盘价 ====')
CSV = _data_path('D049_statsmodels.csv')
import os
if os.path.exists(CSV):
    print(f'命中本地缓存 {CSV},直接读取')
    price = pd.read_csv(CSV, index_col=0, parse_dates=True)
else:
    print('本地无缓存,登录 baostock 联网拉取')
    lg = bs.login()
    if lg.error_code != '0':
        raise RuntimeError(f'baostock 登录失败:{lg.error_msg}')
    series = {name: fetch(code) for name, code in CODES.items()}
    bs.logout()
    # 用 dict 显式映射构造总表(绝不用 df.columns=... 那种按字母序错位的写法)
    price = pd.DataFrame({name: s for name, s in series.items()})
    price.to_csv(CSV)
    print(f'已缓存到 {CSV}')
print(f'对齐前各列长度:{ {k: int(v.notna().sum()) for k, v in price.items()} }')

# ==== 1. 算日收益,按同一张表对齐后再 dropna,避免错位 ====
print('\n==== 1. 算各列日收益,统一对齐 ====')
ret = price.pct_change().dropna()
print(f'对齐后共 {len(ret)} 个交易日,区间 {ret.index[0].date()} ~ {ret.index[-1].date()}')

# ==== 2. 构造 Fama-French 三因子 ====
print('\n==== 2. 构造三因子:市场 MKT / 规模 SMB / 价值 HML ====')
# 无风险利率近似为 0(简化处理,A 股散户日频做演示足够;严谨可减去国债日收益)
MKT = ret['沪深300']                              # 市场因子:大盘本身的涨跌
SMB = ret['中证1000'] - ret['沪深300']            # 规模因子:小盘减大盘
HML = ret['沪深300价值'] - ret['沪深300成长']     # 价值因子:价值减成长
y = ret['隆基绿能']                                # 被解释的对象:隆基绿能的涨跌
print(f'MKT 日均 {MKT.mean()*100:.3f}%  SMB 日均 {SMB.mean()*100:.3f}%  HML 日均 {HML.mean()*100:.3f}%')

# ==== 3. 模型一:单因子 CAPM,只用市场解释隆基 ====
print('\n==== 3. 模型一:单因子 CAPM(隆基 ~ 市场)====')
X1 = sm.add_constant(MKT.rename('MKT'))          # add_constant 加一列 1,用来估计截距(阿尔法)
m1 = sm.OLS(y, X1).fit()                          # OLS 普通最小二乘,fit 一把求出系数
print(m1.summary())
alpha1 = m1.params['const']; beta1 = m1.params['MKT']; r2_1 = m1.rsquared
print(f'\n[单因子] 阿尔法(const)={alpha1*100:.4f}%/天  年化≈{alpha1*TRADE_DAYS*100:.2f}%')
print(f'[单因子] 市场贝塔={beta1:.3f}  R方={r2_1:.4f}  说明 {r2_1*100:.1f}% 的涨跌被大盘解释')

# ==== 4. 模型二:Fama-French 三因子 ====
print('\n==== 4. 模型二:Fama-French 三因子(隆基 ~ 市场+规模+价值)====')
X3 = sm.add_constant(pd.DataFrame({'MKT': MKT, 'SMB': SMB, 'HML': HML}))
m3 = sm.OLS(y, X3).fit()
print(m3.summary())
alpha3 = m3.params['const']; r2_3 = m3.rsquared
b_mkt = m3.params['MKT']; b_smb = m3.params['SMB']; b_hml = m3.params['HML']
t_mkt = m3.tvalues['MKT']; t_smb = m3.tvalues['SMB']; t_hml = m3.tvalues['HML']
print(f'\n[三因子] 阿尔法(const)={alpha3*100:.4f}%/天  年化≈{alpha3*TRADE_DAYS*100:.2f}%')
print(f'[三因子] 市场贝塔={b_mkt:.3f}(t={t_mkt:.2f})  规模={b_smb:.3f}(t={t_smb:.2f})  价值={b_hml:.3f}(t={t_hml:.2f})')
print(f'[三因子] R方={r2_3:.4f}  比单因子的 {r2_1:.4f} {"高" if r2_3>r2_1 else "低"}了 {(r2_3-r2_1)*100:.1f} 个百分点')

# ==== 5. 单因子 vs 三因子 对比小表 ====
print('\n==== 5. 单因子 vs 三因子 对比 ====')
def sig(t):
    return '显著' if abs(t) >= 1.96 else '不显著'
cmp = pd.DataFrame([
    {'指标': 'R方(解释力)', '单因子CAPM': f'{r2_1:.3f}', '三因子': f'{r2_3:.3f}'},
    {'指标': '年化阿尔法', '单因子CAPM': f'{alpha1*TRADE_DAYS*100:.2f}%', '三因子': f'{alpha3*TRADE_DAYS*100:.2f}%'},
    {'指标': '市场贝塔', '单因子CAPM': f'{beta1:.3f}', '三因子': f'{b_mkt:.3f}'},
    {'指标': '规模贝塔(SMB)', '单因子CAPM': '—', '三因子': f'{b_smb:.3f}({sig(t_smb)})'},
    {'指标': '价值贝塔(HML)', '单因子CAPM': '—', '三因子': f'{b_hml:.3f}({sig(t_hml)})'},
])
print(cmp.to_string(index=False))
print('解读:贝塔越大越被大盘放大;阿尔法是扣掉因子后自己的真本事;三因子比单因子能解释更多涨跌')

# ==== 6. 画三张图 ====
print('\n==== 6. 画三张图:净值曲线 / 回归散点 / 三因子系数 ====')

# 图1:隆基 vs 沪深300 累计净值(归一化到 1),看大盘带着走 + 自己更猛
nav = (1 + ret[['隆基绿能', '沪深300']]).cumprod()
nav = nav / nav.iloc[0]
plt.figure(figsize=(11, 6))
plt.plot(nav.index, nav['隆基绿能'], color='#d62728', lw=2, label='隆基绿能')
plt.plot(nav.index, nav['沪深300'], color='#1f77b4', lw=2, label='沪深300(市场)')
plt.title('隆基绿能 vs 沪深300 累计净值:大盘带着走,自己波动更猛')
plt.xlabel('日期'); plt.ylabel('累计净值(起点=1)')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.savefig('nav.png', dpi=120); plt.close()

# 图2:横轴沪深300日收益、纵轴隆基日收益散点 + 单因子回归直线(斜率=贝塔)
plt.figure(figsize=(11, 6))
plt.scatter(MKT, y, s=8, alpha=0.4, color='#9bbbe0', label='每天一个点')
xs = np.linspace(MKT.min(), MKT.max(), 100)
plt.plot(xs, alpha1 + beta1 * xs, color='#d62728', lw=2.2,
         label=f'OLS 回归直线 斜率=贝塔={beta1:.2f}')
plt.axhline(0, color='gray', lw=0.6); plt.axvline(0, color='gray', lw=0.6)
plt.title(f'隆基 vs 沪深300 日收益散点:斜率=贝塔={beta1:.2f}  R方={r2_1:.2f}')
plt.xlabel('沪深300 日收益(大盘)'); plt.ylabel('隆基绿能 日收益')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.savefig('scatter.png', dpi=120); plt.close()

# 图3:三因子系数条形图,标注是否显著
plt.figure(figsize=(11, 6))
names = ['市场 MKT', '规模 SMB', '价值 HML']
vals = [b_mkt, b_smb, b_hml]
tvs = [t_mkt, t_smb, t_hml]
colors = ['#2ca02c' if abs(t) >= 1.96 else '#bbbbbb' for t in tvs]
bars = plt.bar(names, vals, color=colors, width=0.55)
for b, v, t in zip(bars, vals, tvs):
    plt.text(b.get_x() + b.get_width()/2, v + (0.02 if v >= 0 else -0.05),
             f'{v:.2f}\n(t={t:.1f},{sig(t)})', ha='center', fontsize=10)
plt.axhline(0, color='gray', lw=0.8)
plt.title('三因子系数:绿=统计显著(t≥1.96),灰=不显著')
plt.ylabel('回归系数(贝塔)')
plt.grid(alpha=0.3, axis='y'); plt.tight_layout(); plt.savefig('factors.png', dpi=120); plt.close()

print('\n[done] 单因子 + 三因子回归完成,3 张图已生成')

==== 0. 拉隆基绿能 + 4 个指数的日线收盘价 ====
命中本地缓存 /mnt/d/huizi_ai_project/ai_course_video/out/notebook/data/D049_statsmodels.csv,直接读取
对齐前各列长度:{'隆基绿能': 1456, '沪深300': 1456, '中证1000': 1456, '沪深300价值': 1456, '沪深300成长': 1456}

==== 1. 算各列日收益,统一对齐 ====
对齐后共 1455 个交易日,区间 2019-01-03 ~ 2024-12-31

==== 2. 构造三因子:市场 MKT / 规模 SMB / 价值 HML ====
MKT 日均 0.027%  SMB 日均 0.006%  HML 日均 -0.002%

==== 3. 模型一:单因子 CAPM(隆基 ~ 市场)====
                            OLS Regression Results                            
Dep. Variable:                   隆基绿能   R-squared:                       0.247
Model:                            OLS   Adj. R-squared:                  0.246
Method:                 Least Squares   F-statistic:                     476.2
Date:                Thu, 18 Jun 2026   Prob (F-statistic):           1.55e-91
Time:                        17:55:52   Log-Likelihood:                 3217.2
No. Observations:                1455   AIC:                            -6430.
Df Residuals:                    1453   BI

## 真实市场案例

| 市场 | 标的 | 实战观察 |
| --- | --- | --- |
| A股 | sh.601012 | 隆基绿能 2019-2024 日收益对沪深300 做单因子回归,看贝塔(大盘放大倍数)和阿尔法(自己的真本事) |
| 因子构造 | sh.000852 / sh.000300 | 中证1000 减沪深300 造规模因子 SMB,沪深300价值减成长造价值因子 HML,搭出 Fama-French 三因子 |
| 模型对比 | sh.601012 | 单因子 CAPM 和三因子模型跑同一只隆基,比 R方 谁解释得多、阿尔法谁更小,看多加两个因子值不值 |
| 读 summary |  | 看三因子里规模、价值系数的 t 值和 p 值,p 小于 0.05 才当真,否则那股力量对这票没有解释力 |


## 常见坑

### ⚠ 01. 把大盘普涨的浮盈当成自己的真本事

老雷的错就在这:大盘整体上涨时几乎什么都在涨,你赚的那部分大半是贝塔(大盘带的),不是阿尔法(你的水平)。不做回归把两块分开,牛市里人人都觉得自己是股神,行情一回头才发现裸泳。评价自己选股能力,一定要先扣掉大盘的贡献,看剩下的阿尔法是正是负。

### ⚠ 02. 不看 p 值就相信系数

回归会给每个因子算出一个系数,但系数大不代表它真有解释力。一定要看它的 t 值和 p 值:p 值大于 0.05(t 值绝对值小于 2)的系数,很可能只是噪声,不能当真。很多人看到价值因子系数不小就下结论,却忽略它 p 值 0.4 根本不显著,等于在沙子上盖楼。

### ⚠ 03. 用太短的时间窗跑回归

只拿35 个月、几十个交易日的数据跑回归,样本太少,贝塔和阿尔法都会非常不稳,换一段时间结果就大变样。要用足够长的历史(本节用了 6 年),贝塔阿尔法才相对靠谱。尤其阿尔法本来就小,样本太少时几乎没法和噪声区分开。

### ⚠ 04. 忽略阿尔法可能是负的

很多人默认自己的阿尔法是正的,其实扣掉大盘和各因子之后,绝大多数散户的阿尔法是负的,也就是说你折腾半天还跑输了对应风险该有的收益。回归算出来阿尔法是负数并不丢人,看清这个事实,你才会认真考虑是不是干脆买指数更省心。

### ⚠ 05. 把 R方 高低当成好坏

R方 高只说明这票的涨跌被大盘和因子解释得多,绝不代表它能赚钱。一只票完全被大盘牵着走,R方 可以很高,但它可能一直在跌。别把 R方 当成选股的好坏指标。R方 看的是解释力,赚不赚钱要看阿尔法和未来,这是两码事。

## 实战 SOP · 用回归拆解涨跌的几条规矩

1. 评价选股能力先扣大盘:贝塔是大盘带的,阿尔法才是自己的真本事
2. 看系数先看 p 值:p 小于 0.05(t 绝对值大于 2)才当真,否则可能是噪声
3. 跑回归用足够长的样本(本节 6 年),时间太短贝塔阿尔法都不稳
4. 三因子先看各因子是否显著,再解读系数大小,顺序别反
5. R方 看的是解释力不是赚钱能力,别把 R方 高低当选股好坏

> 把这段打印贴在你电脑边,执行 1000 次它会回报你。

## 总结 · 你应该带走的

2. 回归 = 在散点里找一条最贴合的线,斜率告诉你大盘涨 1% 这票大概涨多少。
3. 贝塔是跟大盘的放大倍数(大盘带的),阿尔法是跟大盘无关、自己的真本事。
4. R方 是 0 到 1 的数,告诉你这条线能解释涨跌的几成,比如 0.55 就是 55%。
5. 读 summary 先盯 t 值和 p 值,p 小于 0.05 才当真,再看系数大小。
6. 单因子 CAPM 只用市场,Fama-French 三因子再加规模 SMB 和价值 HML 两股力量。
7. 用 sm.OLS(y, sm.add_constant(x)).fit() 一把求出系数,.summary() 打印全套结果。

## 自测题

**Q1.** 用『找一条线』打比方,解释什么是回归,这条线的斜率和截距各代表什么?

**Q2.** 贝塔和阿尔法有什么区别?老雷为什么会把大盘普涨的浮盈错当成自己的真本事?

**Q3.** 看回归结果时,为什么系数大还不够,一定要看 t 值和 p 值?p 值多小才当真?

**Q4.** Fama-French 三因子是哪三个?规模因子 SMB 和价值因子 HML 分别怎么造出来的?

把答案写下来,3 天后再回看。

## 下一节预告

**Day 050 · scikit-learn 入门** (scikit-learn)

我们用 statsmodels 把涨跌拆成了大盘带的和自己的真本事。下一节我们换一套更强的武器:用 scikit-learn 这个机器学习工具箱,不再只找一条直线,而是喂给程序一堆特征,让它自己学着预测明天是涨还是跌,正式踏进量化里机器学习的大门。

## 推荐阅读

- Sharpe《Capital Asset Prices: A Theory of Market Equilibrium》(1964/Journal of Finance)— 单因子 CAPM 的原始论文,贝塔与阿尔法思想的源头。
- Fama & French《The Cross-Section of Expected Stock Returns》(1992/Journal of Finance)— 提出规模与价值两个因子,三因子模型的奠基之作。
- Fama & French《Common Risk Factors in the Returns on Stocks and Bonds》(1993/Journal of Financial Economics)— 正式给出三因子模型与因子构造方法。
- Andrew Ang《Asset Management: A Systematic Approach to Factor Investing》(2014/Oxford)— 面向普通读者的因子投资科普,把贝塔阿尔法和因子讲得通透。
- Sheppard《Introduction to Python for Econometrics, Statistics and Data Analysis》(2021/在线讲义)— 手把手教用 statsmodels 跑回归读 summary 的实战教程。